# FigureSI_ — Demosaicing vs. direct fit, head-to-head

*Makes up Figures S7–S16.*

Compares localisation precision (`sigma_xy`) and colour precision (`sigma_colour`) for four
fitting-strategy configurations on a Bayer-filtered camera:

| Strategy | Description |
|---|---|
| `STANDARD_DATA` | Direct fit on raw Bayer data, smooth->model->raw-data-weighted refit (current production default) |
| `DEMOSAIC` (bilinear) | Demosaic the puncta ROI with bilinear interpolation, then fit each resulting R/G/B image independently |
| `DEMOSAIC` (malvar) | As above, using Malvar (2004) demosaicing |
| `DEMOSAIC` (menon2007) | As above, using Menon (2007) demosaicing |

For all three `DEMOSAIC` configurations, position (xc, yc) is the photon-weighted mean of the
three channels' fitted positions; total photons = sum of the three channels' photon counts;
per-channel colour = that channel's photon count normalised to the sum. Ground truth
positions are subtracted internally (`subtractx0y0=True`), so `sigma_xy = std(xc)` is
comparable across strategies without a shared ground-truth file.

**`FAST_MODE` flag:** `True` (default) runs ~100 bootstrap fits per condition over ~15
photon levels (log-spaced, 500-50,000) instead of the published 10,000 fits x 200 levels —
same 4 strategies x 3 dyes, so the same comparison, just noisier and coarser, in a few
minutes rather than hours. `FAST_MODE = False` reproduces the published sweep exactly.

In [ ]:
import os
import time
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from scipy.spatial.distance import cdist

from pyS3M import IOFunctions, PlottingBase, sCMOSFunctions, SpectralFunctions, MaskFunctions
import pyS3M.gaussoptfuncs as gaussoptfuncs
from pyS3M.simulation.multicolour import (
    MultiC_Sim_Funcs, FittingStrategy, CameraParameters, SimulationConfig, FittingResultProcessor,
)
# Bare import (not `from src.ImageAnalysisFunctions import ...`): MultiC_Sim_Funcs bare-imports
# ImageAnalysisFunctions internally, so importing it the same way here keeps FittingStrategy
# consistent with what MSF uses. Aliased to avoid clashing with the simulation FittingStrategy above.
from pyS3M.ImageAnalysisFunctions import FittingStrategy as IAF_FittingStrategy

IO = IOFunctions.IO_Functions()
MSF = MultiC_Sim_Funcs()
plotter = PlottingBase.PublicationPlotter(dark_background=False)
sCMOS = sCMOSFunctions.sCMOS_Functions()
S_F = SpectralFunctions.Spectral_Funcs()
M_F = MaskFunctions.Mask_Functions()

REPO_ROOT = Path.cwd().resolve().parents[2]
CALIBRATION_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT = Path("outputs") / "Demosaicing_vs_Fullfit"     # git-ignored
FIGURE_DIR = Path("figures")                              # git-ignored (*.svg)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

FAST_MODE = True  # False -> the published 10,000-fit / 200-photon-level sweep (hours)
print(f"FAST_MODE={FAST_MODE}")
print("Available fitting strategies:", [s.value for s in FittingStrategy])

## Camera calibration and optical setup

In [ ]:
gain = IO.read_tiff(str(CALIBRATION_DIR / "gain.tif"))
offset = IO.read_tiff(str(CALIBRATION_DIR / "offset.tif"))
variance = IO.read_tiff(str(CALIBRATION_DIR / "variance.tif"))
readnoise = IO.read_tiff(str(CALIBRATION_DIR / "readnoise.tif"))
rqe = IO.read_tiff(str(CALIBRATION_DIR / "rqe.tif"))

R, G, B, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])  # Bayer channel order: B, G, R

image_size = 14
masks = M_F.get_masks(size_x=image_size, size_y=image_size)

camera_parameters_dict = {
    "gain": np.full((image_size, image_size), np.median(gain)),
    "offset": np.full((image_size, image_size), np.median(offset)),
    "variance": np.full((image_size, image_size), np.median(variance)),
    "readnoise": np.full((image_size, image_size), np.median(readnoise)),
    "rqe": np.full((image_size, image_size), np.median(rqe)),
    "masks": masks,
    "pixel_QYs": pixel_QYs,
    "pixel_order": ["B", "G", "R"],
    "pixel_order_indices": {"B": 0, "G": 1, "R": 2},
    "mosaic_unit": M_F.mosaic_unit,
}
validated_camera = CameraParameters.validate_and_create(camera_parameters_dict)

import types
smoothing_function = types.SimpleNamespace(
    args={"sigma": 1.5}, extent=1.5,
    smoothing_function=sCMOS.gaussian_filter_stack, data_arg="image",
)

# Optical path filters (same as the Figure 1 dye/PSF panels).
filters = [
    "semrock-di03-r405-488-561-635-t1-25x36",  # dichroic
    "semrock-nf03-405-488-561-635e",            # notch
]
print("Camera parameters validated.")

## Simulation configuration

In [ ]:
n_bootstrap = 100 if FAST_MODE else 10_000
n_photon_steps = 15 if FAST_MODE else 200

base_config = SimulationConfig(
    n_bootstrap=n_bootstrap, background_photons=10.0, background_colour=[1, 1, 1],
    NA=1.49, pixel_size=69, cpu_fraction=0.9, save_raw_results=True,
    subtractx0y0=True,  # ground truth subtracted before saving -- std(xc) = precision
    saverawimages=False, verbose=False, use_stochastic_photons=True, n_unit_cells=7,
)

dyes = ["ATTO 488", "ATTO 565", "ATTO 647N"]
n_photon_space = np.unique(np.around(np.logspace(np.log10(500), np.log10(50000), n_photon_steps) / 5) * 5)

OUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"n_bootstrap={n_bootstrap}  n_photon_space: {len(n_photon_space)} levels, "
      f"{n_photon_space.min():.0f}-{n_photon_space.max():.0f}")
print(f"Results will be saved to: {OUT_ROOT}")

## Example fits: direct vs. demosaic (single punctum)

One simulated punctum (`EXAMPLE_DYE`, 1000 signal photons, 10 background photons/pixel,
`np.random` seed `EXAMPLE_SEED`) fitted by every strategy, with each fit's position and colour
error against the simulation truth. The precision sweep below compares the strategies over many
realisations per condition.

- **Direct fit (`STANDARD_DATA`)**: raw / weights / fit (red X = recovered position) /
  residual / spectrum panels.
- **Demosaic fits (`DEMOSAIC`, bilinear / Malvar / Menon)**: the demosaiced B/G/R images, each
  channel's own fit (red X = that channel's recovered position), and the combined spectral
  fingerprint.

In [ ]:
EXAMPLE_DYE = "ATTO 565"
EXAMPLE_N_PHOTONS = 1000.0
EXAMPLE_BACKGROUND = 10.0
DEMOSAIC_METHODS = {"bilinear": "bilinear", "malvar": "malvar", "menon": "menon2007"}  # figure name -> demosaic_strategy

EXAMPLE_SEED = 27  # np.random seed of the example realisation

avg_wl_ex, dpe_ex = S_F.get_pixel_fractions_dye_and_filters([EXAMPLE_DYE], filters, wavelength, pixel_QYs, normalized=False)
true_colour_ex = np.ravel(dpe_ex) / np.sum(dpe_ex)
emission_ex = {"dye": MSF.filtered_emission_spectrum(EXAMPLE_DYE, filters, wavelength)}

x0_ex = y0_ex = image_size * 69 / 2.0
x0y0_ex = {"dye": np.array([[[x0_ex], [y0_ex]]])}
n_photons_ex = {"dye": np.array([EXAMPLE_N_PHOTONS])}
x_true, y_true = x0_ex / 69, y0_ex / 69  # pixels, as the fitted xc/yc

example_config = replace(base_config, n_bootstrap=1, background_photons=EXAMPLE_BACKGROUND)
x_arr = np.arange(image_size, dtype=np.float32)


def simulate_example(seed):
    np.random.seed(seed)
    bayer, smoothed, _ = MSF.gen_camera_image_stack(
        camera_parameters_dict, wavelength, avg_wl_ex, dpe_ex, n_photons_ex, x0y0_ex,
        smoothing_function=smoothing_function, background_photons=EXAMPLE_BACKGROUND,
        background_colour=[1, 1, 1], NA=1.49, pixel_size=69, emission_spectra=emission_ex,
    )
    if bayer.ndim == 2:
        bayer, smoothed = bayer[np.newaxis], smoothed[np.newaxis]
    return bayer, smoothed


def gaussian_image(xc, yc, s_x, s_y):
    g = np.zeros((image_size, image_size), dtype=np.float32)
    # the model takes (row, col, row width, col width); the fit reports xc = column, yc = row
    return gaussoptfuncs.gaussian_unscaled_model(g, x_arr, image_size, yc, xc, s_y, s_x)


def fit_direct(bayer, smoothed):
    pe, sm, _ = MSF._prepare_fitting_data(bayer, smoothed, validated_camera, FittingStrategy.STANDARD_DATA, example_config)
    weights, _ = MSF._compute_error_maps(sm, None, validated_camera)
    fit = MSF._perform_fitting(FittingStrategy.STANDARD_DATA, pe, sm, weights, None, validated_camera, example_config)
    xc, yc = fit["xc"].iloc[0], fit["yc"].iloc[0]
    A_frac = np.array([fit[f"A_{c}"].iloc[0] for c in validated_camera.pixel_order])
    bg_frac = np.array([fit[f"bg_{c}"].iloc[0] for c in validated_camera.pixel_order])
    shares = {c: float(np.mean(np.asarray(validated_camera.masks[c]) > 0)) for c in validated_camera.pixel_order}
    A_abs = np.array(list(IO.channel_amplitudes(fit.iloc[0], "A_", shares).values()))
    bg_abs = np.array(list(IO.channel_amplitudes(fit.iloc[0], "bg_", shares).values()))
    g = gaussian_image(xc, yc, fit["s_x"].iloc[0], fit["s_y"].iloc[0])
    model = np.zeros((image_size, image_size), dtype=np.float32)
    for i, c in enumerate(validated_camera.pixel_order):
        m = validated_camera.masks[c]
        model[m] = A_abs[i] * g[m] + bg_abs[i]
    return dict(pe=pe[0], weights=weights[0], xc=xc, yc=yc, spectrum=A_frac, model=model, residual=pe[0] - model)


def fit_demosaic(bayer, smoothed, demosaic_strategy):
    cfg = replace(example_config, demosaic_strategy=demosaic_strategy)
    pe, sm, _ = MSF._prepare_fitting_data(bayer, smoothed, validated_camera, FittingStrategy.DEMOSAIC, cfg)
    weights, _ = MSF._compute_error_maps(sm, None, validated_camera)
    fits, _ = MSF.image_analysis.fit_puncta_method(
        [pe[c] for c in range(3)], [sm[c] for c in range(3)], [weights[c] for c in range(3)],
        [(0, 0)] * 3, [0, 1, 2], IAF_FittingStrategy.NOCOLOUR,
    )
    fits = pd.DataFrame(fits, columns=["xc", "yc", "s_x", "s_y", "b", "A", "chi_sqr", "frame"]).sort_values(by="frame").reset_index(drop=True)
    combined = FittingResultProcessor.fit_averager(fits, n_bootstrap=1)
    spectrum = np.array([combined[f"A_{c}"].iloc[0] for c in validated_camera.pixel_order])
    models = [r.A * gaussian_image(r.xc, r.yc, r.s_x, r.s_y) + r.b for r in fits.itertuples()]
    return dict(pe=pe, channel_fits=fits, models=models, spectrum=spectrum,
                xc=combined["xc"].iloc[0], yc=combined["yc"].iloc[0])


def fit_errors(res):
    """(position error / nm, colour error) against the simulation truth; inf if the fit failed."""
    pos = np.hypot(res["xc"] - x_true, res["yc"] - y_true) * 69
    col = np.linalg.norm(res["spectrum"] - true_colour_ex)
    return (pos, col) if np.isfinite(pos) and np.isfinite(col) else (np.inf, np.inf)

In [ ]:
bayer_image_ex, smoothed_image_ex = simulate_example(EXAMPLE_SEED)
direct_ex = fit_direct(bayer_image_ex, smoothed_image_ex)
demosaic_ex = {name: fit_demosaic(bayer_image_ex, smoothed_image_ex, strategy) for name, strategy in DEMOSAIC_METHODS.items()}

errors_ex = {"S3M": fit_errors(direct_ex), **{name: fit_errors(res) for name, res in demosaic_ex.items()}}
print(f"{EXAMPLE_DYE}, {EXAMPLE_N_PHOTONS:.0f} photons, {EXAMPLE_BACKGROUND:.0f} background photons/pixel, seed {EXAMPLE_SEED}")
print(pd.DataFrame(errors_ex, index=["position error / nm", "colour error"]).T.round(4))

In [ ]:
channel_colours = ["#5555ff", "#53b03b", "#ff2a2a"]  # B, G, R
raw_vmin, raw_vmax = np.percentile(direct_ex["pe"], [1, 99])


def spectrum_panel(ax, spectrum):
    ax.bar(np.arange(3), spectrum, color=channel_colours)
    ax.hlines(true_colour_ex, xmin=np.arange(3) - 0.4, xmax=np.arange(3) + 0.4, colors="k", linestyles=":", linewidth=1.5, label="true")
    ax.set_ylim([0, 1]); ax.set_xticks([]); ax.set_yticks([0, 1]); ax.grid(False)
    ax.set_box_aspect(1)  # square, as tall as the images


# Direct fit (S3M): raw / weights / fit (red X = recovered position) / residual / spectrum
fig, axs = plotter.two_column_plot(ncols=5, width_ratios=np.ones(5), width=8.5, height=1.9)
axs[0], _ = plotter.image_plot(axs[0], direct_ex["pe"], vmin=raw_vmin, vmax=raw_vmax, origin="upper", colorbar=True, colorbar_label="photoelectrons")
# the filters of the simulated grid (the simulation's own masks)
PlottingBase.add_bayer_key(axs[0], validated_camera.masks, pixel_order=validated_camera.pixel_order)
axs[1].imshow(direct_ex["weights"], cmap="gray", origin="upper")
axs[1].set_xticks([]); axs[1].set_yticks([])
axs[2], _ = plotter.image_plot(axs[2], direct_ex["model"], vmin=raw_vmin, vmax=raw_vmax, origin="upper", colorbar=True, colorbar_label="photoelectrons")
PlottingBase.add_bayer_key(axs[2], validated_camera.masks, pixel_order=validated_camera.pixel_order)
axs[2].plot(direct_ex["xc"], direct_ex["yc"], "rx", markersize=8, markeredgewidth=2)
rmax = np.max(np.abs(direct_ex["residual"]))
axs[3], _ = plotter.image_plot(axs[3], direct_ex["residual"], cmap="RdBu_r", vmin=-rmax, vmax=rmax, origin="upper", colorbar=True, colorbar_label="photoelectrons")
spectrum_panel(axs[4], direct_ex["spectrum"])
plotter.save_or_show(fig, save_path=str(FIGURE_DIR / "example_fit_standard_data.svg"), show=True)

# Demosaic fits of the same realisation: top row the demosaiced B/G/R images, bottom row each
# channel's fit (red X = that channel's own position); both rows end with the combined spectrum
cmaps = ["Blues", "Greens", "Reds"]
for name, res in demosaic_ex.items():
    fig2, axs2 = plotter.two_column_plot(nrows=2, ncols=5, width_ratios=np.ones(5), height_ratios=[1, 1], width=8.5, height=3.6)
    for row in (0, 1):
        axs2[row, 0], _ = plotter.image_plot(axs2[row, 0], direct_ex["pe"], vmin=raw_vmin, vmax=raw_vmax, origin="upper", colorbar=True, colorbar_label="photoelectrons")
        PlottingBase.add_bayer_key(axs2[row, 0], validated_camera.masks, pixel_order=validated_camera.pixel_order)
        spectrum_panel(axs2[row, 4], res["spectrum"])
    for i, c in enumerate(validated_camera.pixel_order):
        vmin_c, vmax_c = np.percentile(res["pe"][i], [1, 99])
        axs2[0, i + 1], _ = plotter.image_plot(axs2[0, i + 1], res["pe"][i], vmin=vmin_c, vmax=vmax_c, origin="upper", cmap=cmaps[i], colorbar=True, colorbar_label="photoelectrons")
        axs2[1, i + 1], _ = plotter.image_plot(axs2[1, i + 1], res["models"][i], vmin=vmin_c, vmax=vmax_c, origin="upper", cmap=cmaps[i], colorbar=True, colorbar_label="photoelectrons")
        PlottingBase.add_bayer_key(axs2[1, i + 1], validated_camera.masks, pixel_order=validated_camera.pixel_order)
        axs2[1, i + 1].plot(res["channel_fits"]["xc"].iloc[i], res["channel_fits"]["yc"].iloc[i], "rx", markersize=8, markeredgewidth=2)
    plotter.save_or_show(fig2, save_path=str(FIGURE_DIR / f"example_fit_demosaic_{name}.svg"), show=True)

### Timing: direct fit vs. demosaic fit (single punctum)

Wall-clock cost of fitting the one punctum above, direct (`STANDARD_DATA`) vs. demosaic.
Uses the serial `fit_puncta_method` (not the parallel wrapper, which spins up a fresh
`ProcessPoolExecutor` on every call and would swamp the actual per-punctum cost).

In [ ]:
masks_3d = np.dstack([validated_camera.masks[c] for c in validated_camera.pixel_order])
MSF.image_analysis.processors[IAF_FittingStrategy.STANDARD_DATA].readnoise = float(np.median(validated_camera.readnoise))


def time_standard_once():
    t0 = time.perf_counter()
    pe_s, sm_s, _ = MSF._prepare_fitting_data(bayer_image_ex, smoothed_image_ex, validated_camera, FittingStrategy.STANDARD_DATA, example_config)
    w_s, _ = MSF._compute_error_maps(sm_s, None, validated_camera)
    MSF.image_analysis.fit_puncta_method([pe_s[0]], [sm_s[0]], [w_s[0]], [(0, 0)], [0], IAF_FittingStrategy.STANDARD_DATA, masks=[masks_3d])
    return time.perf_counter() - t0


def time_demosaic_once(demosaic_config):
    t0 = time.perf_counter()
    pe_d, sm_d, _ = MSF._prepare_fitting_data(bayer_image_ex, smoothed_image_ex, validated_camera, FittingStrategy.DEMOSAIC, demosaic_config)
    w_d, _ = MSF._compute_error_maps(sm_d, None, validated_camera)
    fits, _ = MSF.image_analysis.fit_puncta_method(
        [pe_d[c] for c in range(3)], [sm_d[c] for c in range(3)], [w_d[c] for c in range(3)],
        [(0, 0)] * 3, [0, 1, 2], IAF_FittingStrategy.NOCOLOUR,
    )
    fits_df = pd.DataFrame(fits, columns=["xc", "yc", "s_x", "s_y", "b", "A", "chi_sqr", "frame"]).sort_values(by="frame").reset_index(drop=True)
    FittingResultProcessor.fit_averager(fits_df, n_bootstrap=1)
    return time.perf_counter() - t0


demosaic_config_bilinear = replace(base_config, n_bootstrap=1, background_photons=EXAMPLE_BACKGROUND, demosaic_strategy="bilinear")
demosaic_config_malvar = replace(base_config, n_bootstrap=1, background_photons=EXAMPLE_BACKGROUND, demosaic_strategy="malvar")
demosaic_config_menon = replace(base_config, n_bootstrap=1, background_photons=EXAMPLE_BACKGROUND, demosaic_strategy="menon2007")

# Warm-up (numba JIT compilation etc.) -- excluded from the timed measurements.
time_standard_once()
time_demosaic_once(demosaic_config_bilinear)
time_demosaic_once(demosaic_config_malvar)
time_demosaic_once(demosaic_config_menon)

N_TIMING_REPEATS = 300 if FAST_MODE else 10_000
standard_times = np.array([time_standard_once() for _ in range(N_TIMING_REPEATS)])
demosaic_times_bilinear = np.array([time_demosaic_once(demosaic_config_bilinear) for _ in range(N_TIMING_REPEATS)])
demosaic_times_malvar = np.array([time_demosaic_once(demosaic_config_malvar) for _ in range(N_TIMING_REPEATS)])
demosaic_times_menon = np.array([time_demosaic_once(demosaic_config_menon) for _ in range(N_TIMING_REPEATS)])

print(f"STANDARD_DATA (direct fit), N={N_TIMING_REPEATS}: {standard_times.mean()*1000:.3f} +/- {standard_times.std()*1000:.3f} ms")
print(f"BILINEAR DEMOSAIC, N={N_TIMING_REPEATS}: {demosaic_times_bilinear.mean()*1000:.3f} +/- {demosaic_times_bilinear.std()*1000:.3f} ms")
print(f"MALVAR DEMOSAIC, N={N_TIMING_REPEATS}: {demosaic_times_malvar.mean()*1000:.3f} +/- {demosaic_times_malvar.std()*1000:.3f} ms")
print(f"MENON DEMOSAIC, N={N_TIMING_REPEATS}: {demosaic_times_menon.mean()*1000:.3f} +/- {demosaic_times_menon.std()*1000:.3f} ms")

In [ ]:
fig3, ax3 = plotter.two_column_plot(ncols=1)

timing_labels = ["Direct fit", "Bilinear Demosaic", "Malvar Demosaic", "Menon Demosaic"]
timing_means_ms = [standard_times.mean() * 1000, demosaic_times_bilinear.mean() * 1000, demosaic_times_malvar.mean() * 1000, demosaic_times_menon.mean() * 1000]
timing_stds_ms = [standard_times.std() * 1000, demosaic_times_bilinear.std() * 1000, demosaic_times_malvar.std() * 1000, demosaic_times_menon.std() * 1000]
ax3.bar(timing_labels, timing_means_ms, yerr=timing_stds_ms, capsize=4, color=["#c96d44", "#7aa456", "#777acd", "#c65999"])
ax3.set_ylabel("time per punctum/ms")
ax3.grid(True, ls="--", alpha=0.5, color="gray")

for i, (mean_ms, std_ms) in enumerate(zip(timing_means_ms, timing_stds_ms)):
    ax3.text(i, mean_ms + std_ms + 0.02 * max(timing_means_ms), f"{mean_ms:.2f} ms", ha="center", va="bottom", fontsize=7)

ax3.set_ylim([0, max(timing_means_ms) * 1.6])
plotter.save_or_show(fig3, save_path=str(FIGURE_DIR / "example_fit_timing.svg"), show=True)

## Run simulations

Four strategy configurations x three dyes. Each configuration uses a distinct
`starting_flag` so files do not collide. `overwrite=False` so an interrupted run can resume.

In [ ]:
runs = [
    (FittingStrategy.STANDARD_DATA, "standard_data_", base_config),
    (FittingStrategy.DEMOSAIC, "demosaic_bilinear_", replace(base_config, demosaic_strategy="bilinear")),
    (FittingStrategy.DEMOSAIC, "demosaic_malvar_", replace(base_config, demosaic_strategy="malvar")),
    (FittingStrategy.DEMOSAIC, "demosaic_menon2007_", replace(base_config, demosaic_strategy="menon2007")),
]

camera_params_dict = validated_camera.__dict__.copy()

for strategy, flag, cfg in runs:
    for dye in dyes:
        label = f"{strategy.value} [{cfg.demosaic_strategy}]" if strategy == FittingStrategy.DEMOSAIC else strategy.value
        print(f"Running {label} / {dye} ...")
        try:
            MSF.test_simulation_method(
                dye=dye, filters=filters, wavelength=wavelength, camera_parameters=camera_params_dict,
                save_folder=str(OUT_ROOT), n_photon_space=n_photon_space, smoothing_function=smoothing_function,
                strategy=strategy, starting_flag=flag, config=cfg, overwrite=False,
            )
            print(f"  Done: {label} / {dye}")
        except Exception as e:
            print(f"  FAILED {label} / {dye}: {e}")

print("All simulations complete.")

## Load results and compute precision metrics

Because `subtractx0y0=True`, the saved `xc`/`yc` columns are already *(fitted - truth)*, so:

`sigma_xy = sqrt((std(xc)^2 + std(yc)^2) / 2) * 69 nm`

Colour precision is the standard deviation of the Euclidean distance from each fitted
`(A_B, A_G, A_R)` point to the true colour vector.

In [ ]:
strategy_labels = ["Direct fit (STANDARD_DATA)", "Demosaic (bilinear)", "Demosaic (malvar)", "Demosaic (menon2007)"]
strategy_flags = ["standard_data_", "demosaic_bilinear_", "demosaic_malvar_", "demosaic_menon2007_"]
parameters = ["xy", "colour", "B", "G", "R", "chi_mean", "chi_median", "chi_std"]

all_files = os.listdir(OUT_ROOT)

Overall_XR = xr.DataArray(
    data=np.full([len(strategy_labels), len(dyes), len(n_photon_space), len(parameters)], np.nan),
    coords=[strategy_labels, dyes, n_photon_space, parameters],
    dims=["Strategy", "Dye", "Photon", "Metric"],
)

for i, (label, flag) in enumerate(zip(strategy_labels, strategy_flags)):
    prefix = flag + "LM_method"
    flag_files = [x for x in all_files if x.startswith(prefix)]
    for j, dye in enumerate(dyes):
        dyestr = dye.replace("/", "-")
        raw_files = [os.path.join(OUT_ROOT, x) for x in flag_files if dyestr in x and "rawresults" in x]
        param_files = [os.path.join(OUT_ROOT, x) for x in flag_files if dyestr in x and "input_parameters" in x]
        if not raw_files or not param_files:
            print(f"Missing files for {label} / {dye} -- skipping.")
            continue

        true_bgr = pd.read_csv(param_files[0]).to_numpy()[0][-3:]
        true_bgr = true_bgr / np.sum(true_bgr)
        true_colour = np.expand_dims(true_bgr, 0)

        results_all = pd.read_hdf(raw_files[0])

        for k, photonval in enumerate(n_photon_space):
            results = results_all[results_all["photon_level"] == k]
            if len(results) == 0:
                continue

            filt = ~((results["s_x"] < 0) | (results["s_y"] < 0))
            xc = results["xc"].to_numpy()[filt]
            yc = results["yc"].to_numpy()[filt]
            chi_sqr = results["chi_sqr"].to_numpy()[filt]

            sigma_xy = np.sqrt((np.nanstd(xc) ** 2 + np.nanstd(yc) ** 2) / 2) * 69

            colour = np.vstack([results["A_B"].to_numpy()[filt], results["A_G"].to_numpy()[filt], results["A_R"].to_numpy()[filt]]).T
            colour_dist = cdist(colour, true_colour).ravel()
            B_dist = np.sqrt(np.square(colour[:, 0] - true_colour[:, 0]))
            G_dist = np.sqrt(np.square(colour[:, 1] - true_colour[:, 1]))
            R_dist = np.sqrt(np.square(colour[:, 2] - true_colour[:, 2]))

            Overall_XR[i, j, k, 0] = sigma_xy
            Overall_XR[i, j, k, 1] = np.nanstd(colour_dist)
            Overall_XR[i, j, k, 2] = np.nanstd(B_dist)
            Overall_XR[i, j, k, 3] = np.nanstd(G_dist)
            Overall_XR[i, j, k, 4] = np.nanstd(R_dist)
            Overall_XR[i, j, k, 5] = np.nanmean(chi_sqr)
            Overall_XR[i, j, k, 6] = np.nanmedian(chi_sqr)
            Overall_XR[i, j, k, 7] = np.nanstd(chi_sqr)

print("Results loaded.")
Overall_XR.to_netcdf(OUT_ROOT / "Demosaic_vs_DirectFit_STDEV.nc")
print("Saved to Demosaic_vs_DirectFit_STDEV.nc")

## Plot — dye-averaged precision vs photon count

In [ ]:
plot_styles = [
    {"color": "#d40000", "linestyle": "-", "linewidth": 1.2},   # Direct fit
    {"color": "black", "linestyle": "-.", "linewidth": 1.0},    # Demosaic (bilinear)
    {"color": "#1f77b4", "linestyle": "--", "linewidth": 1.0},  # Demosaic (malvar)
    {"color": "#2ca02c", "linestyle": ":", "linewidth": 1.2},   # Demosaic (menon2007)
]
plot_labels = ["S$^3$M", "Demosaic (bilinear)", "Demosaic (Malvar)", "Demosaic (Menon)"]

fig, axs = plotter.two_column_plot(ncols=2, width_ratios=np.ones(2))

for i, (label, style) in enumerate(zip(plot_labels, plot_styles)):
    mean_xy = np.nanmean(Overall_XR[i, :, :, 0], axis=0)
    mean_colour = np.nanmean(Overall_XR[i, :, :, 1], axis=0)
    axs[0] = plotter.line_plot(axs[0], n_photon_space, mean_xy, label=label, xlabel=r"$N_{\mathrm{photons}}$", ylabel=r"$\sigma_{xy}$ / nm", **style)
    axs[1] = plotter.line_plot(axs[1], n_photon_space, mean_colour, label=label, xlabel=r"$N_{\mathrm{photons}}$", ylabel=r"$\sigma_{\mathrm{colour}}$ / pixel QE", **style)

for ax in axs:
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim([500, 50000])

axs[0].set_ylim([0.3, 50])
axs[0].set_xlabel("")
axs[1].set_ylim([2e-3, 0.2])
axs[0].legend(loc="best", fontsize=6)

plt.savefig(FIGURE_DIR / "Demosaic_vs_DirectFit.svg", dpi=600, format="svg")
plt.show()

## Plot — per-dye precision vs photon count (one row per dye)

Same as the dye-averaged plot above, but one row per dye instead of averaging over them —
lets strategy differences that are dye-dependent show up directly.

In [ ]:
fig, axs = plotter.two_column_plot(nrows=len(dyes), ncols=2, width_ratios=[1, 1], height_ratios=[1] * len(dyes), width=6.69, height=2.3 * len(dyes))

for row, dye in enumerate(dyes):
    for i, (label, style) in enumerate(zip(plot_labels, plot_styles)):
        xy_vals = Overall_XR[i, row, :, 0]
        colour_vals = Overall_XR[i, row, :, 1]
        axs[row, 0] = plotter.line_plot(axs[row, 0], n_photon_space, xy_vals, label=label, xlabel=r"$N_{\mathrm{photons}}$", ylabel=r"$\sigma_{xy}$ / nm", **style)
        axs[row, 1] = plotter.line_plot(axs[row, 1], n_photon_space, colour_vals, label=label, xlabel=r"$N_{\mathrm{photons}}$", ylabel=r"$\sigma_{\mathrm{colour}}$ / pixel QE", **style)

    axs[row, 0].set_xscale("log"); axs[row, 0].set_yscale("log")
    axs[row, 1].set_xscale("log"); axs[row, 1].set_yscale("log")
    axs[row, 0].set_xlim([500, 50000]); axs[row, 1].set_xlim([500, 50000])
    axs[row, 0].set_ylim([0.3, 50]); axs[row, 1].set_ylim([2e-3, 0.2])
    if row < len(dyes) - 1:
        axs[row, 0].set_xlabel(""); axs[row, 1].set_xlabel("")
    axs[row, 0].text(-0.35, 0.5, dye, transform=axs[row, 0].transAxes, fontsize=8, ha="right", va="center", rotation=90)

axs[0, 0].legend(loc="best", fontsize=6)
plt.savefig(FIGURE_DIR / "Demosaic_vs_DirectFit_per_dye.svg", dpi=600, format="svg")
plt.show()

## Plot — fit-distribution histograms across strategies

One figure per dye, rows = quantity (position bias x/y overlaid; colour fraction B/G/R
overlaid), columns = fitting strategy. `HISTOGRAM_PHOTON_LEVELS` overlays a few photon
counts per panel (colour still encodes the series; linestyle distinguishes photon level).

In [ ]:
HISTOGRAM_PHOTON_LEVELS = [1000.0, 4000.0, 10000.0]
photon_linestyles = ["-", "--", ":", "-."][:len(HISTOGRAM_PHOTON_LEVELS)]
photon_idx = [int(np.argmin(np.abs(n_photon_space - p))) for p in HISTOGRAM_PHOTON_LEVELS]
photon_actual = [n_photon_space[k] for k in photon_idx]

pos_colors = ["tab:orange", "tab:purple"]
col_labels = ["B", "G", "R"]
col_names = ["A_B", "A_G", "A_R"]
hist_bins = 100 if FAST_MODE else 200
col_xlim = (0.0, 1.0)

for dye in dyes:
    dyestr = dye.replace("/", "-")

    raw_by_strategy = []
    for label, flag in zip(strategy_labels, strategy_flags):
        prefix = flag + "LM_method"
        flag_files = [x for x in all_files if x.startswith(prefix)]
        raw_files = [os.path.join(OUT_ROOT, x) for x in flag_files if dyestr in x and "rawresults" in x]
        raw_by_strategy.append(pd.read_hdf(raw_files[0]) if raw_files else None)

    true_colour_dye = None
    for flag in strategy_flags:
        prefix = flag + "LM_method"
        flag_files = [x for x in all_files if x.startswith(prefix)]
        param_files = [os.path.join(OUT_ROOT, x) for x in flag_files if dyestr in x and "input_parameters" in x]
        if param_files:
            true_bgr_dye = pd.read_csv(param_files[0]).to_numpy()[0][-3:]
            true_colour_dye = true_bgr_dye / np.sum(true_bgr_dye)
            break

    pos_concat = []
    for df in raw_by_strategy:
        if df is None:
            continue
        for k in photon_idx:
            sub = df[df["photon_level"] == k]
            pos_concat.append(sub["xc"].to_numpy() * 69)
            pos_concat.append(sub["yc"].to_numpy() * 69)
    pos_concat = np.concatenate(pos_concat) if pos_concat else np.array([0.0])
    pos_concat = pos_concat[np.isfinite(pos_concat)]
    pos_lim = float(np.nanpercentile(np.abs(pos_concat), 99.5)) * 1.2 if len(pos_concat) else 50.0
    pos_xlim = (-pos_lim, pos_lim)

    fig, axs = plotter.two_column_plot(nrows=2, ncols=len(strategy_labels), height=4)

    for col, (label, df) in enumerate(zip(strategy_labels, raw_by_strategy)):
        ax_pos, ax_col = axs[0, col], axs[1, col]

        if df is None:
            ax_pos.axis("off"); ax_col.axis("off")
            continue

        for ls, k, photon_val in zip(photon_linestyles, photon_idx, photon_actual):
            sub = df[df["photon_level"] == k]
            xvals = sub["xc"].to_numpy()[np.isfinite(sub["xc"].to_numpy())] * 69
            yvals = sub["yc"].to_numpy()[np.isfinite(sub["yc"].to_numpy())] * 69
            ax_pos.hist(xvals, bins=hist_bins, range=pos_xlim, density=True, histtype="step",
                        color=pos_colors[0], linestyle=ls, linewidth=0.9, label=f"x ({photon_val:.0f} ph)" if col == 0 else None)
            ax_pos.hist(yvals, bins=hist_bins, range=pos_xlim, density=True, histtype="step",
                        color=pos_colors[1], linestyle=ls, linewidth=0.9, label=f"y ({photon_val:.0f} ph)" if col == 0 else None)

            for cc, cl, cname in zip(channel_colours, col_labels, col_names):
                cvals = sub[cname].to_numpy()
                cvals = cvals[np.isfinite(cvals)]
                ax_col.hist(cvals, bins=hist_bins, range=col_xlim, density=True, histtype="step",
                            color=cc, linestyle=ls, linewidth=0.9, label=f"{cl} ({photon_val:.0f} ph)" if col == 0 else None)

        ax_pos.axvline(0, color="k", linewidth=0.5, linestyle="--")
        if true_colour_dye is not None:
            for cc, tv in zip(channel_colours, true_colour_dye):
                ax_col.axvline(tv, color=cc, linewidth=1.3, linestyle=":", alpha=0.9, zorder=5)

        ax_pos.set_xlim(pos_xlim)
        ax_col.set_xlim(col_xlim)
        ax_col.set_xlabel("Colour fraction", fontsize=7)
        ax_pos.set_xlabel("Position bias / nm", fontsize=7)

        if col == 0:
            ax_pos.set_ylabel("Probability density", fontsize=8)
            ax_col.set_ylabel("Probability density", fontsize=8)
            ax_pos.legend(fontsize=5, loc="upper right", framealpha=0.7, handlelength=1.2, borderpad=0.3, labelspacing=0.2)
            ax_col.legend(fontsize=5, loc="upper right", framealpha=0.7, handlelength=1.2, borderpad=0.3, labelspacing=0.2)
        else:
            ax_pos.set_ylabel(""); ax_col.set_ylabel("")

    fname = f"{dye.replace(' ', '_')}_strategy_histograms.svg"
    fig.savefig(FIGURE_DIR / fname, dpi=600, format="svg", bbox_inches="tight")
    plt.show()
    print(f"Saved: {fname}")

## Next steps

Set `FAST_MODE = False` above and re-run for the published sweep (10,000 bootstrap fits x
200 photon levels x 4 strategies x 3 dyes — expect hours, and a large
`outputs/Demosaicing_vs_Fullfit/` directory of intermediate HDF5/CSV files, git-ignored).